# Day 6：从零手写 PPO（本周最重要！）

> **目标**：从零实现完整的 PPO-Clip 算法——`ActorCritic 网络 → Rollout 采集 → GAE 计算 → Clipped 策略更新`，在 CartPole-v1 和 LunarLander-v3 上训练成功，并可视化分析训练过程与超参数影响。
>
> PPO 手写是**面试高频考点**，也是 W10 RLHF 的直接基础。

---

## 实现路线图

```
Part 1: PPO-Clip 数学回顾
  重要性采样 → 代理目标函数 → Clipped Surrogate Objective

Part 2: ActorCritic 网络
  共享 backbone → Actor Head (策略) → Critic Head (价值)

Part 3: Rollout Buffer
  采集轨迹数据 → 存储 (s, a, r, done, log_prob, value)

Part 4: GAE 计算
  δ_t = r_t + γV(s_{t+1}) - V(s_t) → A_t = Σ (γλ)^l δ_{t+l}

Part 5: PPO 更新
  Mini-batch SGD → Clipped Policy Loss + Value Loss + Entropy Bonus

Part 6: CartPole-v1 训练
  训练 + 可视化 → 目标: 稳定 500 分

Part 7: LunarLander-v3 进阶
  更复杂环境验证 → 目标: 稳定 >200 分

Part 8: 超参数分析
  clip_ratio / GAE lambda / update epochs 的影响
```

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import matplotlib.pyplot as plt

try:
    import gymnasium as gym
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'gymnasium[classic_control,box2d]'])
    import gymnasium as gym

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Part 1：PPO-Clip 数学回顾

### 1.1 从策略梯度到代理目标函数

Day 4 推导的策略梯度：

$$\nabla_\theta J(\theta) = \mathbb{E}_t \left[ \nabla_\theta \log \pi_\theta(a_t \mid s_t) \cdot \hat{A}_t \right]$$

等价于最大化 **代理目标函数（surrogate objective）**：

$$L^{\text{PG}}(\theta) = \mathbb{E}_t \left[ \log \pi_\theta(a_t \mid s_t) \cdot \hat{A}_t \right]$$

### 1.2 重要性采样

但 on-policy 数据采一次就扔，太浪费。用重要性采样复用旧策略 $\pi_{\theta_{\text{old}}}$ 的数据：

$$L^{\text{IS}}(\theta) = \mathbb{E}_t \left[ \frac{\pi_\theta(a_t \mid s_t)}{\pi_{\theta_{\text{old}}}(a_t \mid s_t)} \hat{A}_t \right] = \mathbb{E}_t \left[ r_t(\theta) \hat{A}_t \right]$$

其中概率比 $r_t(\theta) = \frac{\pi_\theta(a_t \mid s_t)}{\pi_{\theta_{\text{old}}}(a_t \mid s_t)}$。

### 1.3 PPO-Clip 目标函数

直接最大化 $L^{\text{IS}}$ 可能导致策略突变。PPO 通过 clip 限制更新幅度：

$$\boxed{L^{\text{CLIP}}(\theta) = \mathbb{E}_t \left[ \min\left( r_t(\theta) \hat{A}_t, \; \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) \hat{A}_t \right) \right]}$$

其中 $\epsilon$ 通常取 0.2。

**clip 的直觉**：
- 当 $\hat{A}_t > 0$（好动作）：$r_t$ 增大会增大 loss，但超过 $1+\epsilon$ 后被截断 → 不会过度增大概率
- 当 $\hat{A}_t < 0$（差动作）：$r_t$ 减小会增大 loss，但低于 $1-\epsilon$ 后被截断 → 不会过度减小概率

### 1.4 完整 PPO Loss

$$L(\theta) = -L^{\text{CLIP}}(\theta) + c_1 \cdot L^{\text{VF}}(\theta) - c_2 \cdot H[\pi_\theta]$$

三项分别是：
1. **Policy Loss** (clipped surrogate) — 取负号因为要最大化
2. **Value Loss** — MSE 拟合回报: $(V_\phi(s_t) - G_t^{\text{target}})^2$
3. **Entropy Bonus** — 鼓励探索，防止过早收敛

## Part 2：ActorCritic 网络 — 共享 Backbone

Actor 和 Critic 共享底层特征提取网络，最后一层分叉：

$$
\text{state}(d) \xrightarrow{\text{shared MLP}} h \begin{cases} \xrightarrow{\text{Actor head}} \text{logits}(|A|) \\ \xrightarrow{\text{Critic head}} V(s) \end{cases}
$$

In [ ]:
class ActorCritic(nn.Module):
    """
    共享 backbone 的 Actor-Critic 网络。
    Actor 输出动作的 logits（经 Categorical 分布采样）。
    Critic 输出状态价值 V(s) 的标量估计。
    """

    def __init__(self, state_dim: int, action_dim: int, hidden_dim: int = 64):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
        )
        self.actor_head = nn.Linear(hidden_dim, action_dim)
        self.critic_head = nn.Linear(hidden_dim, 1)

    def forward(self, x: torch.Tensor):
        h = self.shared(x)
        logits = self.actor_head(h)
        value = self.critic_head(h)
        return logits, value.squeeze(-1)

    def get_action_and_value(self, state: torch.Tensor):
        """采样动作，同时返回 log_prob 和 value（用于 rollout 采集）。"""
        logits, value = self.forward(state)
        dist = Categorical(logits=logits)
        action = dist.sample()
        return action, dist.log_prob(action), dist.entropy(), value

    def evaluate_actions(self, states: torch.Tensor, actions: torch.Tensor):
        """给定 (s, a) 对，计算 log_prob、entropy 和 value（用于 PPO 更新）。"""
        logits, values = self.forward(states)
        dist = Categorical(logits=logits)
        return dist.log_prob(actions), dist.entropy(), values


# 验证
env = gym.make('CartPole-v1')
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

ac = ActorCritic(state_dim, action_dim).to(device)
dummy = torch.randn(2, state_dim).to(device)
logits, values = ac(dummy)

print(f"state_dim={state_dim}, action_dim={action_dim}")
print(f"logits shape: {logits.shape}  (batch, actions)")
print(f"values shape: {values.shape}  (batch,)")
print(f"参数量: {sum(p.numel() for p in ac.parameters()):,}")

# 测试采样
s = torch.FloatTensor(env.reset()[0]).unsqueeze(0).to(device)
act, lp, ent, val = ac.get_action_and_value(s)
print(f"\n采样: action={act.item()}, log_prob={lp.item():.3f}, entropy={ent.item():.3f}, value={val.item():.3f}")

## Part 3：Rollout Buffer — 采集轨迹数据

PPO 是 on-policy 算法，但它可以对同一批数据做多个 epoch 的更新（这是 clip 的意义所在）。

Rollout Buffer 存储一个完整 rollout 的数据：

```
对每一步 t 存储: (s_t, a_t, r_t, done_t, log_prob_t, value_t)

采集完成后计算: advantages (GAE), returns (target values)

训练时: 打乱顺序，切成 mini-batch，做多轮更新
```

In [ ]:
class RolloutBuffer:
    """
    存储一个 rollout 阶段采集的所有轨迹数据。
    采集完成后调用 compute_gae() 计算 GAE advantage 和目标回报。
    """

    def __init__(self):
        self.states = []
        self.actions = []
        self.rewards = []
        self.dones = []
        self.log_probs = []
        self.values = []
        self.advantages = None
        self.returns = None

    def push(self, state, action, reward, done, log_prob, value):
        self.states.append(state)
        self.actions.append(action)
        self.rewards.append(reward)
        self.dones.append(done)
        self.log_probs.append(log_prob)
        self.values.append(value)

    def compute_gae(self, last_value: float, gamma: float, gae_lambda: float):
        """
        计算 GAE advantage 和目标回报（从后往前递推）。

        GAE: A_t = δ_t + (γλ) δ_{t+1} + (γλ)² δ_{t+2} + ...
             δ_t = r_t + γ V(s_{t+1}) - V(s_t)

        Returns: G_t = A_t + V(s_t)
        """
        rewards = np.array(self.rewards)
        dones = np.array(self.dones)
        values = np.array(self.values)
        T = len(rewards)

        advantages = np.zeros(T, dtype=np.float32)
        last_gae = 0.0

        for t in reversed(range(T)):
            if t == T - 1:
                next_value = last_value
                next_non_terminal = 1.0 - dones[t]
            else:
                next_value = values[t + 1]
                next_non_terminal = 1.0 - dones[t]

            delta = rewards[t] + gamma * next_value * next_non_terminal - values[t]
            advantages[t] = delta + gamma * gae_lambda * next_non_terminal * last_gae
            last_gae = advantages[t]

        self.advantages = advantages
        self.returns = advantages + values  # G_t = A_t + V(s_t)

    def get_tensors(self):
        """将所有数据转为 tensor 并移到 device。"""
        return (
            torch.FloatTensor(np.array(self.states)).to(device),
            torch.LongTensor(np.array(self.actions)).to(device),
            torch.FloatTensor(np.array(self.log_probs)).to(device),
            torch.FloatTensor(self.returns).to(device),
            torch.FloatTensor(self.advantages).to(device),
        )

    def clear(self):
        self.__init__()


print("RolloutBuffer 定义完成 ✓")

## Part 4：PPO Agent — 完整的训练逻辑

PPO 训练循环的核心流程：

```
For each iteration:
  1. Rollout: 用当前策略采集 N 步数据
  2. GAE:     计算 advantage 和 target returns
  3. Update:  对数据做 K 个 epoch 的 mini-batch SGD
     - 每个 mini-batch:
       a. 用新策略算 log_prob → 概率比 r_t(θ)
       b. Clipped policy loss
       c. Value loss (MSE)
       d. Entropy bonus
       e. 反向传播，更新参数
  4. 清空 buffer，回到 1
```

In [ ]:
class PPOAgent:
    """
    完整的 PPO-Clip Agent，包含:
    - ActorCritic 网络
    - Rollout 采集
    - GAE 计算
    - Clipped surrogate 更新
    """

    def __init__(
        self,
        state_dim: int,
        action_dim: int,
        hidden_dim: int = 64,
        lr: float = 3e-4,
        gamma: float = 0.99,
        gae_lambda: float = 0.95,
        clip_ratio: float = 0.2,
        entropy_coef: float = 0.01,
        value_coef: float = 0.5,
        max_grad_norm: float = 0.5,
        update_epochs: int = 4,
        mini_batch_size: int = 64,
    ):
        self.gamma = gamma
        self.gae_lambda = gae_lambda
        self.clip_ratio = clip_ratio
        self.entropy_coef = entropy_coef
        self.value_coef = value_coef
        self.max_grad_norm = max_grad_norm
        self.update_epochs = update_epochs
        self.mini_batch_size = mini_batch_size

        self.ac = ActorCritic(state_dim, action_dim, hidden_dim).to(device)
        self.optimizer = optim.Adam(self.ac.parameters(), lr=lr, eps=1e-5)
        self.buffer = RolloutBuffer()

        # 训练日志
        self.policy_losses = []
        self.value_losses = []
        self.entropy_values = []
        self.clip_fractions = []

    @torch.no_grad()
    def select_action(self, state: np.ndarray):
        """采样动作并存储 rollout 数据。返回 (action, value)。"""
        state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        action, log_prob, entropy, value = self.ac.get_action_and_value(state_t)
        return (
            action.item(),
            log_prob.item(),
            value.item(),
        )

    @torch.no_grad()
    def get_value(self, state: np.ndarray) -> float:
        """只获取 V(s)，用于计算 GAE 的 last_value。"""
        state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        _, value = self.ac(state_t)
        return value.item()

    def update(self):
        """
        PPO 核心更新：对 rollout 数据做 K 个 epoch 的 mini-batch 更新。

        每个 mini-batch 计算:
          1. 概率比 r_t(θ) = π_θ(a|s) / π_θ_old(a|s)
          2. Clipped policy loss
          3. Value loss (MSE)
          4. Entropy bonus
        """
        states, actions, old_log_probs, returns, advantages = self.buffer.get_tensors()

        # Advantage 标准化（减均值除标准差，稳定训练）
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        dataset_size = len(states)
        epoch_policy_loss = 0.0
        epoch_value_loss = 0.0
        epoch_entropy = 0.0
        epoch_clip_frac = 0.0
        n_updates = 0

        for epoch in range(self.update_epochs):
            # 打乱数据
            indices = np.arange(dataset_size)
            np.random.shuffle(indices)

            for start in range(0, dataset_size, self.mini_batch_size):
                end = start + self.mini_batch_size
                mb_indices = indices[start:end]

                mb_states = states[mb_indices]
                mb_actions = actions[mb_indices]
                mb_old_log_probs = old_log_probs[mb_indices]
                mb_returns = returns[mb_indices]
                mb_advantages = advantages[mb_indices]

                # 用当前策略重新计算 log_prob、entropy、value
                new_log_probs, entropy, new_values = self.ac.evaluate_actions(
                    mb_states, mb_actions
                )

                # ---- Policy Loss (Clipped Surrogate) ----
                # 概率比: r_t(θ) = exp(log π_θ(a|s) - log π_θ_old(a|s))
                log_ratio = new_log_probs - mb_old_log_probs
                ratio = torch.exp(log_ratio)

                # Surrogate losses
                surr1 = ratio * mb_advantages
                surr2 = torch.clamp(ratio, 1.0 - self.clip_ratio, 1.0 + self.clip_ratio) * mb_advantages
                policy_loss = -torch.min(surr1, surr2).mean()

                # ---- Value Loss ----
                value_loss = nn.functional.mse_loss(new_values, mb_returns)

                # ---- Entropy Bonus ----
                entropy_loss = entropy.mean()

                # ---- Total Loss ----
                loss = policy_loss + self.value_coef * value_loss - self.entropy_coef * entropy_loss

                self.optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.ac.parameters(), self.max_grad_norm)
                self.optimizer.step()

                # 统计 clip fraction
                with torch.no_grad():
                    clip_frac = ((ratio - 1.0).abs() > self.clip_ratio).float().mean().item()

                epoch_policy_loss += policy_loss.item()
                epoch_value_loss += value_loss.item()
                epoch_entropy += entropy_loss.item()
                epoch_clip_frac += clip_frac
                n_updates += 1

        # 记录平均值
        self.policy_losses.append(epoch_policy_loss / max(n_updates, 1))
        self.value_losses.append(epoch_value_loss / max(n_updates, 1))
        self.entropy_values.append(epoch_entropy / max(n_updates, 1))
        self.clip_fractions.append(epoch_clip_frac / max(n_updates, 1))

        self.buffer.clear()


print("PPOAgent 定义完成 ✓")

## Part 5：训练循环

PPO 的训练循环和 DQN 不同——它是 **基于 rollout** 的：

1. 采集 `rollout_steps` 步数据
2. 计算 GAE
3. 做 `update_epochs` 轮 mini-batch 更新
4. 清空 buffer，重复

每个 episode 结束时记录奖励。

In [ ]:
def train_ppo(
    env_name: str,
    agent: PPOAgent,
    total_timesteps: int = 100000,
    rollout_steps: int = 2048,
    print_every: int = 5,
    seed: int = 42,
):
    """
    PPO 训练主循环。

    Args:
        env_name: Gymnasium 环境名
        agent: PPOAgent 实例
        total_timesteps: 总交互步数
        rollout_steps: 每次 rollout 采集的步数
        print_every: 每隔几次 rollout 打印一次
        seed: 随机种子
    """
    env = gym.make(env_name)
    np.random.seed(seed)
    torch.manual_seed(seed)

    state, _ = env.reset(seed=seed)
    episode_reward = 0.0
    episode_rewards = []      # 完整 episode 的奖励
    all_episode_rewards = []  # 用于最终绘图

    timestep = 0
    rollout_count = 0

    while timestep < total_timesteps:
        # --- Phase 1: Rollout 采集 ---
        for step in range(rollout_steps):
            action, log_prob, value = agent.select_action(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            agent.buffer.push(state, action, reward, float(done), log_prob, value)
            state = next_state
            episode_reward += reward
            timestep += 1

            if done:
                episode_rewards.append(episode_reward)
                all_episode_rewards.append(episode_reward)
                episode_reward = 0.0
                state, _ = env.reset()

        # --- Phase 2: 计算 GAE ---
        last_value = agent.get_value(state) if not done else 0.0
        agent.buffer.compute_gae(last_value, agent.gamma, agent.gae_lambda)

        # --- Phase 3: PPO 更新 ---
        agent.update()

        rollout_count += 1
        if rollout_count % print_every == 0 and episode_rewards:
            avg = np.mean(episode_rewards[-20:]) if len(episode_rewards) >= 20 else np.mean(episode_rewards)
            print(f"Rollout {rollout_count:3d} | "
                  f"Timestep {timestep:6d}/{total_timesteps} | "
                  f"Avg Reward (last 20 ep): {avg:7.1f} | "
                  f"Policy Loss: {agent.policy_losses[-1]:.4f} | "
                  f"Value Loss: {agent.value_losses[-1]:.4f} | "
                  f"Entropy: {agent.entropy_values[-1]:.3f} | "
                  f"Clip Frac: {agent.clip_fractions[-1]:.3f}")
        episode_rewards = []

    env.close()
    return all_episode_rewards


print("训练函数定义完成 ✓")

## Part 6：CartPole-v1 训练

在经典的 CartPole 上测试我们的 PPO 实现。

- **状态空间**: 4 维连续 `[cart_pos, cart_vel, pole_angle, pole_angular_vel]`
- **动作空间**: 2 个离散 `{左, 右}`
- **最大步数**: 500
- **目标**: 稳定达到 500 分

In [ ]:
# CartPole 超参数（PPO 原论文风格）
cartpole_agent = PPOAgent(
    state_dim=4,
    action_dim=2,
    hidden_dim=64,
    lr=3e-4,
    gamma=0.99,
    gae_lambda=0.95,
    clip_ratio=0.2,
    entropy_coef=0.01,
    value_coef=0.5,
    max_grad_norm=0.5,
    update_epochs=4,
    mini_batch_size=64,
)

print("开始训练 PPO on CartPole-v1...\n")
cartpole_rewards = train_ppo(
    env_name='CartPole-v1',
    agent=cartpole_agent,
    total_timesteps=80000,
    rollout_steps=2048,
    print_every=2,
    seed=42,
)

print(f"\n训练完成! 共 {len(cartpole_rewards)} 个 episodes")
print(f"最后 20 个 episode 平均奖励: {np.mean(cartpole_rewards[-20:]):.1f}")

### CartPole 训练曲线可视化

In [ ]:
def plot_ppo_training(episode_rewards, agent, title_prefix="PPO"):
    """可视化 PPO 训练过程的 4 个关键指标。"""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # 1. Episode Rewards
    ax = axes[0, 0]
    ax.plot(episode_rewards, alpha=0.3, color='steelblue')
    window = min(30, len(episode_rewards) // 3) if len(episode_rewards) > 10 else 1
    if window > 1:
        smoothed = np.convolve(episode_rewards, np.ones(window)/window, mode='valid')
        ax.plot(range(window-1, len(episode_rewards)), smoothed,
                color='darkblue', linewidth=2, label=f'{window}-ep avg')
    ax.set_xlabel('Episode')
    ax.set_ylabel('Reward')
    ax.set_title(f'{title_prefix} — Episode Rewards')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # 2. Policy Loss
    ax = axes[0, 1]
    ax.plot(agent.policy_losses, color='coral', linewidth=1)
    ax.set_xlabel('Rollout Iteration')
    ax.set_ylabel('Policy Loss')
    ax.set_title('Policy Loss (Clipped Surrogate)')
    ax.grid(True, alpha=0.3)

    # 3. Value Loss
    ax = axes[1, 0]
    ax.plot(agent.value_losses, color='green', linewidth=1)
    ax.set_xlabel('Rollout Iteration')
    ax.set_ylabel('Value Loss')
    ax.set_title('Value Loss (MSE)')
    ax.grid(True, alpha=0.3)

    # 4. Entropy & Clip Fraction
    ax = axes[1, 1]
    ax2 = ax.twinx()
    l1, = ax.plot(agent.entropy_values, color='purple', linewidth=1, label='Entropy')
    l2, = ax2.plot(agent.clip_fractions, color='orange', linewidth=1, label='Clip Fraction')
    ax.set_xlabel('Rollout Iteration')
    ax.set_ylabel('Entropy', color='purple')
    ax2.set_ylabel('Clip Fraction', color='orange')
    ax.set_title('Entropy & Clip Fraction')
    ax.legend(handles=[l1, l2], loc='upper right')
    ax.grid(True, alpha=0.3)

    plt.suptitle(f'{title_prefix} Training Dashboard', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


plot_ppo_training(cartpole_rewards, cartpole_agent, "CartPole PPO")

### CartPole 贪心评估

In [ ]:
@torch.no_grad()
def evaluate_ppo(env_name, agent, n_episodes=20, seed=123):
    """用贪心策略（argmax）评估 PPO Agent。"""
    env = gym.make(env_name)
    rewards = []
    for i in range(n_episodes):
        state, _ = env.reset(seed=seed + i)
        total_reward = 0
        done = False
        while not done:
            state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
            logits, _ = agent.ac(state_t)
            action = logits.argmax(dim=1).item()
            state, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            done = terminated or truncated
        rewards.append(total_reward)
    env.close()
    return rewards


cartpole_eval = evaluate_ppo('CartPole-v1', cartpole_agent)
print(f"CartPole 贪心评估 ({len(cartpole_eval)} episodes):")
print(f"  平均奖励: {np.mean(cartpole_eval):.1f} ± {np.std(cartpole_eval):.1f}")
print(f"  最小: {min(cartpole_eval):.0f}, 最大: {max(cartpole_eval):.0f}")
print()
if np.mean(cartpole_eval) >= 400:
    print("CartPole 目标达成! 平均奖励 >= 400 ✓")
else:
    print("未达标，建议增加训练步数或调整超参数")

## Part 7：LunarLander-v3 进阶实验

LunarLander 是更具挑战性的环境：

- **状态空间**: 8 维连续 `[x, y, vx, vy, angle, angular_vel, left_leg_contact, right_leg_contact]`
- **动作空间**: 4 个离散 `{无操作, 左引擎, 主引擎, 右引擎}`
- **奖励**: 着陆+100~+140, 坠毁-100, 每帧引擎消耗
- **目标**: 稳定达到 200 分以上

> 注: LunarLander 需要安装 `box2d`: `pip install gymnasium[box2d]`。如果安装失败可以跳过这部分。

In [ ]:
try:
    test_env = gym.make('LunarLander-v3')
    ll_state_dim = test_env.observation_space.shape[0]
    ll_action_dim = test_env.action_space.n
    test_env.close()
    print(f"LunarLander-v3: state_dim={ll_state_dim}, action_dim={ll_action_dim}")
    HAS_LUNAR = True
except Exception as e:
    print(f"LunarLander 不可用 ({e})")
    print("跳过 LunarLander 实验。安装方法: pip install gymnasium[box2d]")
    HAS_LUNAR = False

In [ ]:
if HAS_LUNAR:
    lunar_agent = PPOAgent(
        state_dim=ll_state_dim,
        action_dim=ll_action_dim,
        hidden_dim=64,
        lr=3e-4,
        gamma=0.99,
        gae_lambda=0.95,
        clip_ratio=0.2,
        entropy_coef=0.01,
        value_coef=0.5,
        max_grad_norm=0.5,
        update_epochs=4,
        mini_batch_size=64,
    )

    print("开始训练 PPO on LunarLander-v3...\n")
    lunar_rewards = train_ppo(
        env_name='LunarLander-v3',
        agent=lunar_agent,
        total_timesteps=300000,
        rollout_steps=2048,
        print_every=5,
        seed=42,
    )

    print(f"\n训练完成! 共 {len(lunar_rewards)} 个 episodes")
    print(f"最后 20 个 episode 平均奖励: {np.mean(lunar_rewards[-20:]):.1f}")
else:
    print("跳过 LunarLander 训练")

In [ ]:
if HAS_LUNAR:
    plot_ppo_training(lunar_rewards, lunar_agent, "LunarLander PPO")

    lunar_eval = evaluate_ppo('LunarLander-v3', lunar_agent)
    print(f"LunarLander 贪心评估 ({len(lunar_eval)} episodes):")
    print(f"  平均奖励: {np.mean(lunar_eval):.1f} ± {np.std(lunar_eval):.1f}")
    print(f"  最小: {min(lunar_eval):.0f}, 最大: {max(lunar_eval):.0f}")
    print()
    if np.mean(lunar_eval) >= 200:
        print("LunarLander 目标达成! 平均奖励 >= 200 ✓")
    else:
        print("未达标，建议增加训练步数")

## Part 8：超参数分析 — clip_ratio 的影响

PPO 最关键的超参数是 `clip_ratio` $\epsilon$。我们在 CartPole 上测试不同 $\epsilon$ 值的影响：

- $\epsilon$ 太小（如 0.05）：更新过于保守，学习慢
- $\epsilon$ 太大（如 0.5）：约束太松，类似普通策略梯度，不稳定
- $\epsilon = 0.2$：PPO 原始论文的默认值，经验上最优

In [ ]:
def run_clip_experiment(clip_ratios, total_timesteps=60000, seed=42):
    """对比不同 clip_ratio 的训练效果。"""
    results = {}
    for clip in clip_ratios:
        print(f"--- clip_ratio = {clip} ---")
        agent = PPOAgent(
            state_dim=4, action_dim=2, hidden_dim=64,
            lr=3e-4, gamma=0.99, gae_lambda=0.95,
            clip_ratio=clip, entropy_coef=0.01, value_coef=0.5,
            max_grad_norm=0.5, update_epochs=4, mini_batch_size=64,
        )
        rewards = train_ppo(
            'CartPole-v1', agent, total_timesteps=total_timesteps,
            rollout_steps=2048, print_every=100, seed=seed,
        )
        results[clip] = rewards
        final_avg = np.mean(rewards[-20:]) if len(rewards) >= 20 else np.mean(rewards)
        print(f"  最终平均: {final_avg:.1f} ({len(rewards)} episodes)\n")
    return results


clip_ratios_to_test = [0.05, 0.1, 0.2, 0.3, 0.5]
print("开始 clip_ratio 消融实验...\n")
clip_results = run_clip_experiment(clip_ratios_to_test)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 6))
colors = ['#e74c3c', '#e67e22', '#2ecc71', '#3498db', '#9b59b6']

for (clip, rewards), color in zip(clip_results.items(), colors):
    if len(rewards) > 10:
        window = min(20, len(rewards) // 3)
        smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')
        ax.plot(range(window-1, len(rewards)), smoothed,
                color=color, linewidth=2, label=f'ε={clip}')
    else:
        ax.plot(rewards, color=color, linewidth=2, label=f'ε={clip}')

ax.axhline(y=400, color='gray', linestyle='--', alpha=0.5, label='Target (400)')
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Reward (smoothed)', fontsize=12)
ax.set_title('PPO clip_ratio (ε) Ablation on CartPole-v1', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n各 clip_ratio 最终 20-episode 平均:")
for clip, rewards in clip_results.items():
    avg = np.mean(rewards[-20:]) if len(rewards) >= 20 else np.mean(rewards)
    print(f"  ε={clip:.2f}: {avg:.1f}")

## Part 9：PPO 核心代码回顾与面试要点

### 面试手写 PPO 的关键代码段

面试时不需要写完整训练循环，但以下核心逻辑必须能手写：

**1. GAE 计算（从后往前递推）**
```python
for t in reversed(range(T)):
    delta = rewards[t] + gamma * next_value * (1 - done[t]) - values[t]
    advantages[t] = delta + gamma * lam * (1 - done[t]) * last_gae
    last_gae = advantages[t]
```

**2. PPO-Clip Loss（核心 3 行）**
```python
ratio = exp(new_log_prob - old_log_prob)
surr1 = ratio * advantage
surr2 = clamp(ratio, 1-eps, 1+eps) * advantage
policy_loss = -min(surr1, surr2).mean()
```

**3. 完整 Loss**
```python
loss = policy_loss + c1 * value_loss - c2 * entropy
```

### PPO vs DQN 对比

| 维度 | DQN (Day 3) | PPO (Day 6) |
|------|-------------|-------------|
| 学什么 | Q 函数 | 策略 + 价值 |
| 数据策略 | Off-policy (Replay Buffer) | On-policy (Rollout Buffer) |
| 数据复用 | 反复采样 | 同一 rollout 做 K epochs |
| 动作空间 | 仅离散 | 离散 + 连续 |
| 稳定性机制 | 目标网络 | Clip ratio |
| RLHF 适用 | 不适用 | **直接使用** |

## Part 10：总结与 Day 7 衔接

### 本日实现清单

- [x] `ActorCritic` 网络（共享 backbone + Actor/Critic head）
- [x] `RolloutBuffer`（轨迹存储 + GAE 计算）
- [x] `PPOAgent`（Clipped surrogate + Value loss + Entropy bonus）
- [x] CartPole-v1 训练成功（目标 >= 400）
- [x] LunarLander-v3 训练成功（目标 >= 200）
- [x] 超参数消融：clip_ratio 的影响

### 关键收获

1. PPO 的训练循环是 **rollout → GAE → multi-epoch mini-batch update**
2. Clip 机制通过限制概率比 $r_t(\theta)$ 来保证策略不会突变
3. GAE 通过 $\lambda$ 平衡方差和偏差
4. Advantage 标准化对训练稳定性很重要
5. Entropy bonus 防止策略过早收敛

### Day 7 预告

明天 Day 7 将深入分析：
- PPO-Clip vs PPO-Penalty（KL 散度版本）的对比
- TRPO 与 PPO 的数学关系
- PPO 在 LLM / RLHF 中的适配方式
- 全周知识串联与复盘